In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the train dataset
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\ml_benchmark\\04_titanic\\split_train.csv'
train_df = pd.read_csv(train_data_path)

# Display the first few rows of the dataset
print(train_df.head())

# Check the data types and basic information
print(train_df.info())

# Check for missing values
print(train_df.isnull().sum())

# Summary statistics for numerical columns
print(train_df.describe())

# Summary statistics for categorical columns
print(train_df.describe(include=['O']))

# Visualize the distribution of the target variable
sns.countplot(x='Survived', data=train_df)
plt.title('Survival Count')
plt.show()

# Visualize the distribution of numerical features
numerical_features = train_df.select_dtypes(include=[np.number]).columns.tolist()
for feature in numerical_features:
    plt.figure()
    sns.histplot(train_df[feature], kde=True)
    plt.title(f'Distribution of {feature}')
    plt.show()

# Visualize the distribution of categorical features
categorical_features = train_df.select_dtypes(include=['O']).columns.tolist()
for feature in categorical_features:
    plt.figure()
    sns.countplot(x=feature, data=train_df)
    plt.title(f'Distribution of {feature}')
    plt.xticks(rotation=45)
    plt.show()

# Correlation matrix for numerical features
correlation_matrix = train_df[numerical_features].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


   PassengerId  Survived  Pclass  ...      Fare Cabin  Embarked
0          409         0       3  ...    7.7750   NaN         S
1          481         0       3  ...   46.9000   NaN         S
2          511         1       3  ...    7.7500   NaN         Q
3          610         1       1  ...  153.4625  C125         S
4          548         1       2  ...   13.8625   NaN         C

[5 rows x 12 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Survived     712 non-null    int64  
 2   Pclass       712 non-null    int64  
 3   Name         712 non-null    object 
 4   Sex          712 non-null    object 
 5   Age          567 non-null    float64
 6   SibSp        712 non-null    int64  
 7   Parch        712 non-null    int64  
 8   Ticket       712 non-null    object 
 9   Fare         712 non-

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-08-30 15:55:59.779 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], 'Numeric': ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale
from metagpt.tools.libs.feature_engineering import TargetMeanEncoder
import pandas as pd

# Load the evaluation dataset
eval_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\ml_benchmark\\04_titanic\\split_eval.csv'
eval_df = pd.read_csv(eval_data_path)

# Copy the datasets to avoid modifying the original data
train_df_copy = train_df.copy()
eval_df_copy = eval_df.copy()

# Handle missing values
# For numerical features, use the median
num_features = ['Age', 'Fare']
fill_missing_num = FillMissingValue(num_features, strategy='median')
train_df_copy = fill_missing_num.fit_transform(train_df_copy)
eval_df_copy = fill_missing_num.transform(eval_df_copy)

# For categorical features, use the most frequent value
cat_features = ['Embarked', 'Cabin']
fill_missing_cat = FillMissingValue(cat_features, strategy='most_frequent')
train_df_copy = fill_missing_cat.fit_transform(train_df_copy)
eval_df_copy = fill_missing_cat.transform(eval_df_copy)

# Encode categorical variables
# Use target mean encoding for 'Sex' and 'Embarked'
target_mean_encoder_sex = TargetMeanEncoder('Sex', 'Survived')
train_df_copy = target_mean_encoder_sex.fit_transform(train_df_copy)
eval_df_copy = target_mean_encoder_sex.transform(eval_df_copy)

target_mean_encoder_embarked = TargetMeanEncoder('Embarked', 'Survived')
train_df_copy = target_mean_encoder_embarked.fit_transform(train_df_copy)
eval_df_copy = target_mean_encoder_embarked.transform(eval_df_copy)

# For 'Cabin', extract the first letter and use target mean encoding
train_df_copy['Cabin'] = train_df_copy['Cabin'].str[0]
eval_df_copy['Cabin'] = eval_df_copy['Cabin'].str[0]

target_mean_encoder_cabin = TargetMeanEncoder('Cabin', 'Survived')
train_df_copy = target_mean_encoder_cabin.fit_transform(train_df_copy)
eval_df_copy = target_mean_encoder_cabin.transform(eval_df_copy)

# Normalize numerical features
num_features_to_scale = ['Age', 'Fare']
scaler = StandardScale(num_features_to_scale)
train_df_copy = scaler.fit_transform(train_df_copy)
eval_df_copy = scaler.transform(eval_df_copy)

# Display the first few rows of the preprocessed datasets
print(train_df_copy.head())
print(eval_df_copy.head())


   PassengerId  Survived  ...  Embarked_target_mean Cabin_target_mean
0          409         0  ...              0.331407          0.318966
1          481         0  ...              0.331407          0.318966
2          511         1  ...              0.362069          0.318966
3          610         1  ...              0.331407          0.531915
4          548         1  ...              0.548148          0.318966

[5 rows x 15 columns]
   PassengerId  Survived  ...  Embarked_target_mean Cabin_target_mean
0          206         0  ...              0.331407          0.500000
1           45         1  ...              0.362069          0.318966
2          822         1  ...              0.331407          0.318966
3          459         1  ...              0.331407          0.318966
4          796         0  ...              0.331407          0.318966

[5 rows x 15 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_copy)
print("column_info")
print(column_info)


column_info
{'Category': ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], 'Numeric': ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_target_mean', 'Embarked_target_mean', 'Cabin_target_mean'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from metagpt.tools.libs.data_preprocess import LabelEncode
import pandas as pd

# Label encode non-numeric columns
non_numeric_features = ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
label_encoder = LabelEncode(non_numeric_features)
train_df_copy = label_encoder.fit_transform(train_df_copy)
eval_df_copy = label_encoder.transform(eval_df_copy)

# Prepare the data for training
X_train = train_df_copy.drop(['Survived', 'PassengerId'], axis=1)
y_train = train_df_copy['Survived']
X_eval = eval_df_copy.drop(['Survived', 'PassengerId'], axis=1)
y_eval = eval_df_copy['Survived']

# Initialize the XGBoost classifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2],
}

# Perform grid search to find the best parameters
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Predict on the evaluation set
y_pred = best_model.predict(X_eval)

# Calculate the accuracy
accuracy = accuracy_score(y_eval, y_pred)
print(f'Accuracy on the evaluation set: {accuracy:.4f}')


Accuracy on the evaluation set: 0.6983


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [15:57:12] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
